In [0]:
df = spark.table("workspace.default.vf_hackathon_dataset_india_large").toPandas()
print(df.shape)
df.head(3)

print(df.columns.tolist())
print(df.iloc[0].to_dict())


claim_keywords = {
    "icu":        ["icu", "intensive care", "critical care", "icu bed", "intensive care unit", "critical care unit", "icu facility", "icu services"],
    "surgery":    ["surgery", "surgical", "operation", "operative", "theatre", "surgical procedure", "surgical services", "operation theatre", "surgical center"],
    "anesthesia": ["anesthesia", "anaesthesia", "anesthetic", "anaesthetic", "anesthesia services", "anesthesia department", "anesthesia care", "anesthesiology"],
    "oxygen":     ["oxygen", "o2", "oxygen supply", "oxygen cylinder", "oxygen therapy", "medical oxygen", "oxygen concentrator", "oxygen facility"],
    "ventilator": ["ventilator", "mechanical ventilation", "ventilatory support", "ventilator support", "ventilator services", "ventilator facility", "ventilation equipment"]
}

conflict_rules = [
    ("surgery", "anesthesia"),   # Surgery ohne Anesthesia = Konflikt
    ("ventilator", "icu"),       # Ventilator ohne ICU = verdächtig
    ("surgery", "oxygen"),       # Surgery ohne Oxygen = Konflikt
    ("icu", "oxygen"),           # ICU ohne Oxygen = Konflikt
    ("icu", "ventilator"),       # ICU ohne Ventilator = verdächtig
    ("anesthesia", "oxygen"),    # Anesthesia ohne Oxygen = Konflikt
    ("ventilator", "oxygen"),    # Ventilator ohne Oxygen = Konflikt
    ("surgery", "icu"),          # Surgery ohne ICU = verdächtig
    ("anesthesia", "icu"),       # Anesthesia ohne ICU = verdächtig
]


def calc_trust_score(claims: dict, conflicts: list) -> float:
    base = sum(claims.values()) / len(claims)
    penalty = len(conflicts) * 0.15
    return round(max(0.0, base - penalty), 2)